# 01 - BBOB Problem Noise Landscape Analysis

### 🔬 Central Research Question:
> **"How does stochastic noise injection distort underlying fitness landscape topology, gradient signals, and global minimum accessibility across continuous optimization benchmarks?"**

This notebook provides a **rigorous mathematical and empirical analysis of noise injection models** across representative BBOB continuous benchmark functions:
- **$f_1$ (Sphere):** Separable unimodal bowl.
- **$f_8$ (Rosenbrock):** Ill-conditioned curved valley.
- **$f_{11}$ (Discus):** High conditioning single sensitive direction.
- **$f_{15}$ (Rastrigin Multi-Modal):** Highly multimodal grid with strong global structure.
- **$f_{21}$ (Gallagher 101 Peaks):** 101 asymmetric Gaussian peaks with $2\text{–}5$ unit local elevation gaps.

---

### 📐 Mathematical Formulations of Compared Noise Models:
1. **Deterministic Clean Ground Truth:**
   $$f_{\text{clean}}(x)$$

2. **Naive Multiplicative Noise (Global Magnitude Scaling):**
   $$\tilde{f}_{\text{naive}}(x) = f(x) + \mathcal{N}\left(0, \left(\sigma \cdot |f(x)|\right)^2\right)$$
   *Limitation:* Because variance scales with the raw magnitude $|f(x)|$, even when the optimizer reaches the true basin (where $f(x) \approx 40$ on $f_{21}$), noise standard deviation remains huge ($\approx 4.0$), completely swallowing subtle $2\text{–}5$ unit local peaks and confusing convergence verification.

3. **Custom Optimality-Gap Multiplicative Noise (Heteroscedastic Gap Scaling):**
   $$\tilde{f}_{\text{opt\_gap}}(x) = f(x) + \mathcal{N}\left(0, \left(\sigma \cdot |f(x) - f_{\text{opt}}|\right)^2\right)$$
   *Advantage:* Noise standard deviation scales strictly with distance-to-optimum in objective space. Far from the target, high variance actively challenges global exploration; as $x \to x^*$, $|f(x) - f_{\text{opt}}| \to 0$, smoothly restoring the deterministic basin geometry at the global minimum.


In [8]:
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path('.').resolve()
root_dir = cwd.parent if cwd.name == 'notebooks' else cwd
src_dir = root_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Setup paths and imports
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import rankdata

# Add project root src/ to sys.path

from shared.config import DATA_DIR, RESULTS_DIR, PROJECT_ROOT
from evolution.infra.problems.bbob import BBOBProblem
from evolution.domain.services.noise_strategy import (
    HeteroscedasticNoiseStrategy,
    NoNoiseStrategy,
    HomoscedasticAdditiveNoiseStrategy,
    AWGNStrategy,
)
from benchmarking.domain.taxonomy import BBOB_NAMES, BBOB_CLASSES

pub_dir = RESULTS_DIR / 'publication' / 'noise_landscapes'
pub_dir.mkdir(parents=True, exist_ok=True)

print('✅ Environment initialized with BBOB problem suites and noise strategy engines.')


## 1. Unified Experiment Configuration
Define test problems, dimensions, and noise levels. Set `TARGET_PROBLEM_IDS` to generate figures across all benchmark classes.

In [9]:
TARGET_PROBLEM_IDS = [1, 8, 11, 15, 21]  # All 5 core benchmark problems
dim = 2
instance_id = 1
primary_noise_std = 0.10
noisy_levels = [0.05, 0.1, 0.2]
all_levels = [0.0] + noisy_levels

# Global aesthetic color palette
color_palette = {
    0.0:  '#0F172A',  # slate dark (clean)
    0.05: '#10B981',  # emerald green
    0.1:  '#06B6D4',  # cyan
    0.2:  '#3B82F6',  # blue
    0.5:  '#8B5CF6',  # purple
    'naive': '#EF4444', # red (naive noise)
    'gap':   '#10B981'  # emerald (optimality-gap noise)
}

print(f"🎯 Target Problems Configured ({len(TARGET_PROBLEM_IDS)}): {[f'f{p}: {BBOB_NAMES.get(p, "Function")} ({BBOB_CLASSES.get(p, "Class")})' for p in TARGET_PROBLEM_IDS]}")
print(f"⚡ Primary Noise Level for 3-Panel Demonstrations: σ = {primary_noise_std}")


## 2. 1D Cross-Section 3-Panel Noise Comparison Across Functions

### 🎯 Why we do this plot:
Instead of abstract line plots of variance numbers, this **3-Panel Methodology Comparison** directly visualizes continuous 1D cross-sections passing through the global optimum of each BBOB function under different noise regimes.

### 📖 How to read the 3 Panels:
- **Panel A (Clean Landscape):** Continuous deterministic function cross-section, showing the exact bowl, valley, or multimodal peaks and the global basin.
- **Panel B (Naive Multiplicative Noise, $\sigma = 0.1$):** Magnitude-based noise $\mathcal{N}(0, (0.1 \cdot |f(x)|)^2)$. Because variance scales with $|f(x)|$, high-cost regions and even valleys are overwhelmed by huge point clouds, burying delicate landscape features.
- **Panel C (Optimality-Gap Noise, $\sigma = 0.1$):** Gap-based model $\mathcal{N}(0, (0.1 \cdot |f(x) - f_{\text{opt}}|)^2)$. High noise variance exists at domain boundaries for active exploration, but smoothly funnels to **exact zero variance** at $x^*$, preserving the global basin geometry.

In [10]:
n_pts = 400
t = np.linspace(-5.0, 5.0, n_pts)
n_repeats = 6
t_scatter = np.repeat(t, n_repeats)

for p_id in TARGET_PROBLEM_IDS:
    problem = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=NoNoiseStrategy(), instance_id=instance_id)
    f_opt = problem.true_optimum
    x_opt = problem.optimum_x
    p_name = BBOB_NAMES.get(p_id, problem.name)
    p_class = BBOB_CLASSES.get(p_id, 'Benchmark')
    fn_slug = f"f{p_id}_{problem.name.lower()}"
    fn_dir = pub_dir / fn_slug
    fn_dir.mkdir(parents=True, exist_ok=True)

    # 1D slice through optimum along dimension 0
    pts_1d = np.tile(x_opt, (n_pts, 1))
    pts_1d[:, 0] = t
    y_clean = np.array([problem(pt) for pt in pts_1d])

    pts_scatter = np.repeat(pts_1d, n_repeats, axis=0)
    y_clean_rep = np.repeat(y_clean, n_repeats)

    # 1. Naive Multiplicative Noise: noise_std * |f(x)|
    np.random.seed(42)
    naive_std_curve = primary_noise_std * np.abs(y_clean)
    y_naive_scatter = y_clean_rep + np.random.normal(0, primary_noise_std * np.abs(y_clean_rep), size=len(y_clean_rep))

    # 2. Optimality-Gap Heteroscedastic Noise: noise_std * |f(x) - f_opt|
    p_gap = BBOBProblem(problem_id=p_id, dim=dim, instance_id=instance_id, noise_strategy=HeteroscedasticNoiseStrategy(noise_std=primary_noise_std))
    np.random.seed(42)
    gap_std_curve = primary_noise_std * np.abs(y_clean - f_opt)
    y_gap_scatter = np.array([p_gap(pt) for pt in pts_scatter])

    fig_3p = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            f'<b>(A) Clean f_{p_id} ({p_name})</b><br><span style="font-size:11px;color:#64748B;">{p_class} Landscape</span>',
            f'<b>(B) Naive Multiplicative (σ={primary_noise_std})</b><br><span style="font-size:11px;color:#EF4444;">Variance scales with |f(x)|</span>',
            f'<b>(C) Optimality-Gap (σ={primary_noise_std})</b><br><span style="font-size:11px;color:#10B981;">Preserved zero variance at f_opt</span>'
        ],
        horizontal_spacing=0.07
    )

    # Panel A
    fig_3p.add_trace(go.Scatter(x=t, y=y_clean, mode='lines', name=f'Clean f_{p_id}', line=dict(color='#0F172A', width=2.8)), row=1, col=1)
    fig_3p.add_trace(go.Scatter(x=[x_opt[0]], y=[f_opt], mode='markers+text', text=[f' f_opt={f_opt:.2f}'], textposition='top right',
                                name='Global Optimum (x*)', marker=dict(color='#EF4444', size=11, symbol='star', line=dict(width=1, color='#FFFFFF'))), row=1, col=1)

    # Panel B
    fig_3p.add_trace(go.Scatter(x=np.concatenate([t, t[::-1]]), y=np.concatenate([y_clean + 2 * naive_std_curve, (y_clean - 2 * naive_std_curve)[::-1]]),
                                fill='toself', fillcolor='rgba(239, 68, 68, 0.15)', line=dict(color='rgba(255,255,255,0)'), name='±2σ Naive Band'), row=1, col=2)
    fig_3p.add_trace(go.Scatter(x=t_scatter, y=y_naive_scatter, mode='markers', name='Naive Evals', marker=dict(color='#EF4444', size=3.5, opacity=0.45)), row=1, col=2)
    fig_3p.add_trace(go.Scatter(x=t, y=y_clean, mode='lines', name='Clean Ref', line=dict(color='#0F172A', width=1.5, dash='dash')), row=1, col=2)

    # Panel C
    fig_3p.add_trace(go.Scatter(x=np.concatenate([t, t[::-1]]), y=np.concatenate([y_clean + 2 * gap_std_curve, (y_clean - 2 * gap_std_curve)[::-1]]),
                                fill='toself', fillcolor='rgba(16, 185, 129, 0.20)', line=dict(color='rgba(255,255,255,0)'), name='±2σ Gap Band'), row=1, col=3)
    fig_3p.add_trace(go.Scatter(x=t_scatter, y=y_gap_scatter, mode='markers', name='Gap Evals', marker=dict(color='#10B981', size=3.5, opacity=0.45)), row=1, col=3)
    fig_3p.add_trace(go.Scatter(x=t, y=y_clean, mode='lines', name='Clean Ref', line=dict(color='#0F172A', width=1.5, dash='dash'), showlegend=False), row=1, col=3)
    fig_3p.add_trace(go.Scatter(x=[x_opt[0]], y=[f_opt], mode='markers', name='Zero Variance at x*', marker=dict(color='#047857', size=11, symbol='star', line=dict(width=1, color='#FFFFFF'))), row=1, col=3)

    fig_3p.update_xaxes(title_text='Spatial Coordinate x₁ (x₂ = x₂*)', showgrid=True, gridcolor='#F1F5F9', linecolor='#CBD5E1')
    fig_3p.update_yaxes(title_text='Objective f(x)', showgrid=True, gridcolor='#F1F5F9', linecolor='#CBD5E1', row=1, col=1)
    fig_3p.update_yaxes(showgrid=True, gridcolor='#F1F5F9', linecolor='#CBD5E1', row=1, col=2)
    fig_3p.update_yaxes(showgrid=True, gridcolor='#F1F5F9', linecolor='#CBD5E1', row=1, col=3)

    for anno in fig_3p.layout.annotations:
        anno.update(font=dict(size=12, color='#0F172A', family='Inter, sans-serif'), yshift=6)

    fig_3p.update_layout(
        title=dict(
            text=f'<b>Noise Model Distortion (1D Cross-Section) — BBOB f_{p_id} ({p_name}, {p_class})</b><br>' +
                 '<span style="font-size:12px;color:#64748B;">Deterministic vs. Naive Magnitude Noise vs. Preserved Optimality-Gap Basin Structure</span>',
            x=0.01, xanchor='left', y=0.98,
            font=dict(size=14, color='#0F172A', family='Inter, Helvetica, sans-serif')
        ),
        height=480, width=1240,
        margin=dict(l=65, r=40, t=95, b=60),
        template='plotly_white',
        legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center', bgcolor='rgba(255,255,255,0.95)', bordercolor='#E2E8F0', borderwidth=1)
    )

    out_p = fn_dir / 'figure_1d_noise_cross_section.png'
    fig_3p.write_image(str(out_p), scale=3)
    print(f'✅ Exported 1D 3-panel figure: {fn_slug}/figure_1d_noise_cross_section.png')


## 3. 2D Contour & Basin Topology Comparison Across Functions

### 🎯 Why we do this plot:
To observe how 2D contour level sets and local basins distort across the 2D spatial search domain under the three noise regimes for every function class.

In [11]:
grid_resol = 70
x_range = np.linspace(-5.0, 5.0, grid_resol)
y_range = np.linspace(-5.0, 5.0, grid_resol)
X, Y = np.meshgrid(x_range, y_range)

def to_rank_norm(Z):
    zf = Z.flatten()
    zr = rankdata(zf)
    return ((zr - zr.min()) / (zr.max() - zr.min())).reshape(Z.shape)

for p_id in TARGET_PROBLEM_IDS:
    problem = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=NoNoiseStrategy(), instance_id=instance_id)
    f_opt = problem.true_optimum
    x_opt = problem.optimum_x
    p_name = BBOB_NAMES.get(p_id, problem.name)
    p_class = BBOB_CLASSES.get(p_id, 'Benchmark')
    fn_slug = f"f{p_id}_{problem.name.lower()}"
    fn_dir = pub_dir / fn_slug
    fn_dir.mkdir(parents=True, exist_ok=True)

    Z_clean = np.zeros_like(X)
    for i in range(grid_resol):
        for j in range(grid_resol):
            Z_clean[i, j] = float(problem([X[i, j], Y[i, j]]))

    np.random.seed(42)
    Z_naive = Z_clean + np.random.normal(0, primary_noise_std * np.abs(Z_clean))
    np.random.seed(42)
    Z_gap = Z_clean + np.random.normal(0, primary_noise_std * np.abs(Z_clean - f_opt))

    fig_contour_3p = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            f'<b>(A) Clean f_{p_id} ({p_name})</b><br><span style="font-size:11px;color:#64748B;">{p_class} Deterministic</span>',
            f'<b>(B) Naive Multiplicative (σ={primary_noise_std})</b><br><span style="font-size:11px;color:#EF4444;">Fractured level sets</span>',
            f'<b>(C) Optimality-Gap (σ={primary_noise_std})</b><br><span style="font-size:11px;color:#10B981;">Preserved central basin</span>'
        ],
        horizontal_spacing=0.06
    )

    grids = [('Clean', to_rank_norm(Z_clean)), ('Naive', to_rank_norm(Z_naive)), ('Optimality Gap', to_rank_norm(Z_gap))]
    for idx, (label, z_grid) in enumerate(grids, start=1):
        fig_contour_3p.add_trace(
            go.Contour(
                x=x_range, y=y_range, z=z_grid,
                colorscale='Viridis',
                showscale=(idx == 3),
                colorbar=dict(title='Rank Norm', thickness=14, len=0.8) if idx == 3 else None,
                contours=dict(coloring='heatmap', showlines=True)
            ),
            row=1, col=idx
        )
        fig_contour_3p.add_trace(
            go.Scatter(
                x=[x_opt[0]], y=[x_opt[1]], mode='markers',
                marker=dict(color='#FFFFFF', symbol='x', size=10, line=dict(width=2.5, color='#EF4444')),
                showlegend=False, name='Global Min x*'
            ),
            row=1, col=idx
        )

    for anno in fig_contour_3p.layout.annotations:
        anno.update(font=dict(size=12, color='#0F172A', family='Inter, sans-serif'), yshift=6)

    fig_contour_3p.update_xaxes(title_text='x₁', showgrid=True, gridcolor='#E2E8F0', linecolor='#94A3B8')
    fig_contour_3p.update_yaxes(title_text='x₂', showgrid=True, gridcolor='#E2E8F0', linecolor='#94A3B8')

    fig_contour_3p.update_layout(
        title=dict(
            text=f'<b>2D Topology Distortion: Clean vs. Naive vs. Optimality-Gap on BBOB f_{p_id} ({p_name})</b>',
            x=0.01, xanchor='left', y=0.98,
            font=dict(size=14, color='#0F172A', family='Inter, Helvetica, sans-serif')
        ),
        height=430, width=1240,
        margin=dict(l=60, r=60, t=85, b=40),
        template='plotly_white'
    )

    out_p = fn_dir / 'figure_2d_noise_topology.png'
    fig_contour_3p.write_image(str(out_p), scale=3)
    print(f'✅ Exported 2D 3-panel figure: {fn_slug}/figure_2d_noise_topology.png')


## 4. 3D Surface Landscape Topology Comparison Across Functions

### 🏔️ 3D Perspective Visualizations
We render 3-panel 3D surface plots for all target benchmark functions ($f_1, f_8, f_{11}, f_{15}, f_{21}$):
- **Panel A (Clean 3D Surface):** Unperturbed mathematical topology showing true hills, valleys, and global optimum basin.
- **Panel B (Naive Multiplicative Noise):** High jagged variance spikes throughout the search space, fracturing the surface continuity.
- **Panel C (Optimality-Gap Heteroscedastic Noise):** Boundary noise gradients with a smooth, preserved central basin around $x^*$.

Exports high-resolution figures to `results/publication/noise_landscapes/{fn_slug}/figure_3d_noise_surface.png`.

In [12]:
grid_resol_3d = 50
x_range_3d = np.linspace(-5.0, 5.0, grid_resol_3d)
y_range_3d = np.linspace(-5.0, 5.0, grid_resol_3d)
X_3d, Y_3d = np.meshgrid(x_range_3d, y_range_3d)

camera_view = dict(
    eye=dict(x=1.5, y=-1.5, z=1.3),
    center=dict(x=0, y=0, z=-0.1)
)

for p_id in TARGET_PROBLEM_IDS:
    problem = BBOBProblem(problem_id=p_id, dim=dim, noise_strategy=NoNoiseStrategy(), instance_id=instance_id)
    f_opt = problem.true_optimum
    x_opt = problem.optimum_x
    p_name = BBOB_NAMES.get(p_id, problem.name)
    p_class = BBOB_CLASSES.get(p_id, 'Benchmark')
    fn_slug = f"f{p_id}_{problem.name.lower()}"
    fn_dir = pub_dir / fn_slug
    fn_dir.mkdir(parents=True, exist_ok=True)

    Z_clean = np.zeros_like(X_3d)
    for i in range(grid_resol_3d):
        for j in range(grid_resol_3d):
            Z_clean[i, j] = float(problem([X_3d[i, j], Y_3d[i, j]]))

    np.random.seed(42)
    Z_naive = Z_clean + np.random.normal(0, primary_noise_std * np.abs(Z_clean))
    np.random.seed(42)
    Z_gap = Z_clean + np.random.normal(0, primary_noise_std * np.abs(Z_clean - f_opt))

    fig_3d = make_subplots(
        rows=1, cols=3,
        specs=[[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}]],
        subplot_titles=[
            f'<b>(A) Clean 3D Surface (f_{p_id})</b><br><span style="font-size:11px;color:#64748B;">{p_class} Deterministic</span>',
            f'<b>(B) Naive Multiplicative (σ={primary_noise_std})</b><br><span style="font-size:11px;color:#EF4444;">Magnitude-scaled surface spikes</span>',
            f'<b>(C) Heteroscedastic Gap (σ={primary_noise_std})</b><br><span style="font-size:11px;color:#10B981;">Smooth preserved basin at f_opt</span>'
        ],
        horizontal_spacing=0.04
    )

    # Clean Surface
    fig_3d.add_trace(go.Surface(x=X_3d, y=Y_3d, z=Z_clean, colorscale='Viridis', showscale=False, opacity=0.95), row=1, col=1)
    # Naive Surface
    fig_3d.add_trace(go.Surface(x=X_3d, y=Y_3d, z=Z_naive, colorscale='Reds', showscale=False, opacity=0.90), row=1, col=2)
    # Heteroscedastic Surface
    fig_3d.add_trace(go.Surface(x=X_3d, y=Y_3d, z=Z_gap, colorscale='Viridis', showscale=False, opacity=0.95), row=1, col=3)

    fig_3d.update_layout(
        title=dict(
            text=f'<b>3D Fitness Landscape Topology: Clean vs. Naive vs. Heteroscedastic on BBOB f_{p_id} ({p_name})</b>',
            x=0.01, xanchor='left', y=0.98,
            font=dict(size=14, color='#0F172A', family='Inter, Helvetica, sans-serif')
        ),
        scene=dict(camera=camera_view, xaxis_title='x₁', yaxis_title='x₂', zaxis_title='f(x)'),
        scene2=dict(camera=camera_view, xaxis_title='x₁', yaxis_title='x₂', zaxis_title='f(x)'),
        scene3=dict(camera=camera_view, xaxis_title='x₁', yaxis_title='x₂', zaxis_title='f(x)'),
        height=480,
        width=1380,
        margin=dict(l=30, r=30, t=80, b=30),
        template='plotly_white'
    )

    out_p = fn_dir / 'figure_3d_noise_surface.png'
    fig_3d.write_image(str(out_p), scale=3)
    print(f'✅ Exported 3D 3-panel surface figure: {fn_slug}/{out_p.name}')
